# 08 — MLOps Demo
**CRISP-DM Phase 6: Deployment & Monitoring**

Demonstrates the production MLOps workflow:
1. MLflow experiment tracking & model registry
2. Batch scoring pipeline
3. Drift monitoring with Evidently + PSI
4. API serving (FastAPI)
5. Docker orchestration

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

from churn.config import cfg
print('Config loaded. MLflow URI:', cfg.mlflow.tracking_uri)

Config loaded. MLflow URI: mlruns


## 1. MLflow Experiment Tracking

In [2]:
import mlflow

mlflow.set_tracking_uri(cfg.mlflow.tracking_uri)
mlflow.set_experiment(cfg.mlflow.experiment_name)

# Demo: log a run using real artefacts when available
artefacts_dir = Path(cfg.paths.model_artefacts_dir)
metrics_path = artefacts_dir / 'metrics_test.json'
best_params_path = artefacts_dir / 'best_params.json'

metrics = {
    'auc_pr': 0.4180,
    'auc_roc': 0.8711,
    'brier_score': 0.0557,
    'precision_at_10pct': 0.4350,
}
if metrics_path.exists():
    with open(metrics_path) as f:
        loaded = json.load(f)
    metrics = {
        'auc_pr': float(loaded.get('auc_pr', metrics['auc_pr'])),
        'auc_roc': float(loaded.get('auc_roc', metrics['auc_roc'])),
        'brier_score': float(loaded.get('brier_score', metrics['brier_score'])),
        'precision_at_10pct': float(loaded.get('precision_at_10pct', metrics['precision_at_10pct'])),
    }

with mlflow.start_run(run_name='demo-notebook-run') as run:
    mlflow.log_params({'model_type': 'lightgbm', 'tuning': 'optuna'})
    mlflow.log_metrics(metrics)

    if best_params_path.exists():
        mlflow.log_artifact(str(best_params_path))
    if metrics_path.exists():
        mlflow.log_artifact(str(metrics_path))

    print(f'Run ID: {run.info.run_id}')
    print(f'Experiment: {cfg.mlflow.experiment_name}')
    print('Logged metrics:', {k: round(v, 4) for k, v in metrics.items()})

/Users/victoroko/Documents/tech-test/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/victoroko/Documents/tech-test/.venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Run ID: 558dd97b34c74726a1d2b7301ac89d66
Experiment: churn-prediction
Logged metrics: {'auc_pr': 0.418, 'auc_roc': 0.8711, 'brier_score': 0.0557, 'precision_at_10pct': 0.435}


## 2. Model Registry

In [3]:
# Register model (in production, trainer.py does this automatically)
client = mlflow.tracking.MlflowClient()

# List registered models
try:
    models = client.search_registered_models()
    for m in models:
        print(f'Model: {m.name}')
        for v in m.latest_versions:
            print(f'  Version {v.version} — Stage: {v.current_stage}')
except Exception as e:
    print(f'No registered models yet: {e}')
    print('Run the full training pipeline to register a model.')

/Users/victoroko/Documents/tech-test/.venv/lib/python3.11/site-packages/mlflow/tracking/_model_registry/utils.py:220: FutureWarning: The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri)


## 3. Batch Scoring

In [2]:
from churn.scoring.batch_scorer import score_active_customers

# Score all active customers
try:
    scored = score_active_customers()
    print(f'Scored {len(scored):,} customers')
    print('\nRisk tier distribution:')
    display(scored['risk_tier'].value_counts().to_frame('count'))
    print('\nTop 10 highest risk:')
    display(scored.head(10)[['unique_customer_identifier', 'churn_probability', 'risk_tier', 'driver_1', 'driver_2']])
except FileNotFoundError as e:
    print(f'Model not yet trained: {e}')
    print('Run notebook 04_modelling.ipynb first.')

/Users/victoroko/Documents/tech-test/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/victoroko/Documents/tech-test/.venv/lib/python3.11/site-packages/pandera/_pandas_deprecated.py:146: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this

Scored 203,275 customers

Risk tier distribution:


,count
risk_tier,
Low,199429
Medium,3512
High,334



Top 10 highest risk:


,unique_customer_identifier,churn_probability,risk_tier,driver_1,driver_2
0,0a7601cdf05e374b192a84fd6ba29fad2044954bd470dd...,0.8,High,contract_dd_cancels,speed_gap_pct
1,c885847f327da2efe91db8b02bfc737628c456addea74e...,0.8,High,contract_dd_cancels,speed_gap_pct
2,a7c01fab18d75c1fcf5265a603cf4192f6a88624d4614a...,0.8,High,contract_dd_cancels,speed_gap_pct
3,342e36c41c8fb8e9f796cad2be22cd77fc7f63bfc27311...,0.8,High,contract_dd_cancels,speed_gap_pct
4,4d48f13b53d3b80dc171f05e620e489fd3a9f272e3ea73...,0.8,High,contract_dd_cancels,speed_gap_pct
5,c885847f327da2efe91db8b02bfc737628c456addea74e...,0.8,High,contract_dd_cancels,speed_gap_pct
6,c8fda57ab0052e1f0ab996a52084b3d8c51e9a3b1f1be4...,0.8,High,contract_dd_cancels,speed_gap_pct
7,9393cb284d3d7b0c6dea23853fab303bcf108e0b0ba5bb...,0.8,High,contract_dd_cancels,dd_cancel_60_day
8,b67dc3a43ed8689f548ebcb4a90e40c429b4615e780643...,0.8,High,contract_dd_cancels,dd_cancel_60_day
9,3c7e17d49ede506ba6fa7aec72cb064d97403b72b7f332...,0.8,High,contract_dd_cancels,speed_gap_pct


## 4. Drift Monitoring

In [5]:
from churn.monitoring.drift_detector import run_full_monitoring
from churn.monitoring.psi_calculator import compute_psi_all_features

# Simulate drift by comparing train vs test snapshots
from churn.data.splitter import temporal_split

features_dir = Path(cfg.paths.features_dir)
parquets = sorted(features_dir.glob('features_*.parquet'))

if len(parquets) >= 2:
    fm = pd.concat([pd.read_parquet(p) for p in parquets], ignore_index=True)
    split = temporal_split(fm)

    # PSI between train and test
    psi_results = compute_psi_all_features(split.train, split.test)
    psi_df = pd.DataFrame(psi_results.items(), columns=['feature', 'psi'])
    print('PSI Results (top features by PSI):')
    display(psi_df.sort_values('psi', ascending=False).head(10))

    # Evidently report
    result = run_full_monitoring(split.train, split.test)
    print(f'\nDrift detected: {result["drift_detected"]}')
    if result.get('psi_alerts'):
        print(f'PSI alerts: {result["psi_alerts"]}')
else:
    print('Need at least 2 feature matrices. Run notebook 03 first.')

PSI ALERT: days_since_last_call = 0.3768 (threshold=0.20)


PSI Results (top features by PSI):


,feature,psi
25,days_since_last_call,0.3768
36,sales_channel_encoded,0.1147
26,call_frequency_trend,0.0603
2,days_to_ooc,0.0472
3,contract_status_risk,0.0220
24,avg_hold_time_30d,0.0166
22,pct_loyalty_calls_90d,0.0153
21,loyalty_call_count_90d,0.0099
9,contract_dd_cancels,0.0096
1,is_out_of_contract,0.0095


Evidently HTML export not supported in current version.
PSI ALERT: days_since_last_call = 0.3768 (threshold=0.20)
PSI ALERT — features with PSI > 0.2: ['days_since_last_call']



Drift detected: False
PSI alerts: ['days_since_last_call']


## 5. API Serving Demo

In [6]:
# Show API schema
from churn.api.schemas import CustomerFeatures, PredictionResponse
import json

print('Request schema:')
print(json.dumps(CustomerFeatures.model_json_schema(), indent=2)[:500])

print('\nResponse schema:')
print(json.dumps(PredictionResponse.model_json_schema(), indent=2))

Request schema:
{
  "description": "Input features for a single customer prediction.",
  "examples": [
    {
      "avg_download_30d": 850.0,
      "avg_talk_time_30d": 420.0,
      "call_count_30d": 4,
      "contract_status_risk": 3,
      "dd_cancel_60_day": 1,
      "download_trend_7_30": 0.6,
      "loyalty_call_flag_30d": 1,
      "ooc_days": 15,
      "speed_gap_pct": 0.15,
      "tenure_days": 730
    }
  ],
  "properties": {
    "ooc_days": {
      "description": "Days out of contract (negative = in co

Response schema:
{
  "description": "Output from /predict endpoint.",
  "properties": {
    "churn_probability": {
      "maximum": 1,
      "minimum": 0,
      "title": "Churn Probability",
      "type": "number"
    },
    "risk_tier": {
      "description": "High / Medium / Low",
      "title": "Risk Tier",
      "type": "string"
    },
    "top_drivers": {
      "description": "Top SHAP drivers",
      "items": {
        "type": "string"
      },
      "title": "Top Drivers

In [7]:
# To start the API locally:
# !uvicorn churn.api.main:app --reload --port 8000

# Or via Docker:
# !cd docker && docker compose up -d api

# Example curl:
print('''
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{
    "call_count_30d": 5,
    "loyalty_call_flag_30d": 1,
    "avg_download_30d": 45.2,
    "download_trend_7_30": -0.15,
    "ooc_days": 120,
    "speed_gap_pct": 0.25,
    "contract_status_risk": 3,
    "tenure_days": 800,
    "avg_talk_time_30d": 240
  }'
''')


curl -X POST http://localhost:8000/predict   -H "Content-Type: application/json"   -d '{
    "call_count_30d": 5,
    "loyalty_call_flag_30d": 1,
    "avg_download_30d": 45.2,
    "download_trend_7_30": -0.15,
    "ooc_days": 120,
    "speed_gap_pct": 0.25,
    "contract_status_risk": 3,
    "tenure_days": 800,
    "avg_talk_time_30d": 240
  }'



## 6. Architecture Overview

```
┌─────────────────────────────────────────────────────────────┐
│                    Production Architecture                   │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌──────────┐   ┌──────────┐   ┌───────────┐              │
│  │  Raw Data │──▶│ DuckDB   │──▶│ Feature   │              │
│  │  (S3/GCS)│   │ Ingest   │   │ Store     │              │
│  └──────────┘   └──────────┘   └─────┬─────┘              │
│                                       │                     │
│                              ┌────────▼────────┐            │
│                              │  LightGBM Train  │            │
│                              │  (Optuna + CV)   │            │
│                              └────────┬────────┘            │
│                                       │                     │
│                    ┌──────────────────┼──────────┐          │
│                    ▼                  ▼          ▼          │
│              ┌──────────┐    ┌──────────┐ ┌──────────┐     │
│              │ MLflow    │    │ Batch    │ │ FastAPI  │     │
│              │ Registry  │    │ Scorer   │ │ /predict │     │
│              └──────────┘    └────┬─────┘ └──────────┘     │
│                                   │                         │
│                    ┌──────────────┼──────────────┐          │
│                    ▼              ▼              ▼          │
│              ┌──────────┐  ┌──────────┐  ┌──────────┐     │
│              │ Evidently │  │ Alerts*  │  │ scored   │     │
│              │ Drift Mon │  │ (Slack)  │  │ .csv     │     │
│              └──────────┘  └──────────┘  └──────────┘     │
│                                                             │
│  Orchestration: Airflow DAGs (daily/weekly/6-hourly)       │
│  Infra in repo: Docker Compose (local)                      │
│  * Slack hooks are scaffolded in DAGs and need webhook env │
└─────────────────────────────────────────────────────────────┘
```